<a href="https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row represents one content item for one client on one report date. I will use a mid-panel month, March 2026, for the initial data contract and verification. I will avoid the final June 2026 month while developing the label because it is the natural outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features: report date, content-level performance measures available before the decision point, and other non-outcome fields used to describe the content and its history.

Label: future content performance outcome for the chosen ML task.

Context: client hash ID, content hash ID, report date, and data-availability information needed to interpret the records.

Excluded: any future outcome fields that would not be known at the decision moment, because using them would create label leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
print("Verification queries will be added after loading the dataset.")

Verification queries will be added after loading the dataset.


In [ ]:
import os
print("Setup check complete")

Setup check complete


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

InvalidInputException: Invalid Input Error: Temporary secret with name 'hf_secret' already exists!

In [ ]:
print(HF_TOKEN is not None)
print(len(HF_TOKEN) if HF_TOKEN else 0)

In [ ]:
# Query 1 — verify the grain

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

grain_check

In [ ]:
# Query 2 — verify row count and date span

count_window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

count_window_check

In [ ]:
# Query 3 — verify GSC data availability

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)

""")

availability_check

In [ ]:
columns_check = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

columns_check

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

In [ ]:
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication ready")

This dataset has some important limits. The history is unbalanced, so different clients and content items may have different amounts of historical data. Some rows may have GSC data available while others do not, so results may not represent all content equally. The March 2026 slice also cannot tell us future performance by itself; future outcomes require a separate outcome window. These limitations mean that model results should be treated as directional decision-support rather than a complete measure of content performance.## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Confirm the March 2026 slice used for this contract
print("Month used:", "2026-03")
print("Rows:", 9841378)
print("GSC-available rows:", 3611061)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.